# landing → bronze: 年齢各歳・男女別人口(050008_toukeika_032)

SageMaker Unified Studio の JupyterLab(Python 3 カーネル)で上から順に実行する。
データは合計 3.3MB なので Spark は使わず、boto3 だけで処理する。

1. landing の CSV が UTF-8 であることを確かめて BOM を外す(UTF-8 以外は変換せず失敗にする)
2. 改行コードを LF にそろえる
3. 構造を確認する(全行の列数がヘッダと同じか / 空行・注釈行が混ざっていないか)
4. **全ファイルが確認を通ったときだけ** bronze に書き出す

| 層 | 場所 |
| --- | --- |
| landing | `s3://<bucket>/akita/landing/population_by_age/` |
| bronze | `s3://<bucket>/akita/bronze/population_by_age/` |
| 確認結果 | `s3://<bucket>/akita/_reports/population_by_age/` |
| 一時置き場 | `s3://<bucket>/akita/_staging/population_by_age/<実行時刻>/` |

確認結果と一時置き場を bronze の下に置かないのは、Athena がテーブルの場所にあるファイルを
拡張子に関係なく全部データとして読むため。

出典: 「年齢別人口流動調査（県・市町村の年齢各歳、男女別人口）」(秋田県)
秋田県オープンデータカタログサイト, CC BY 4.0
https://ckan.pref.akita.lg.jp/dataset/050008_toukeika_032

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────────────────
import os

# True のあいだは確認だけして書き込まない。結果を見てから False にして再実行する。
DRY_RUN = True

REGION = "ap-northeast-1"
BUCKET_PREFIX = "jaws-festa-2026"
LANDING_PREFIX = "akita/landing/population_by_age/"
BRONZE_PREFIX = "akita/bronze/population_by_age/"
REPORT_PREFIX = "akita/_reports/population_by_age/"
STAGING_PREFIX = "akita/_staging/population_by_age/"

# 1列目(年月日コード)の形。これに合わない行は注釈行の疑いとして扱う。
KEY_PATTERN = r"^\d{10}$"

In [ ]:
import boto3

session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE"), region_name=REGION)
s3 = session.client("s3")
account_id = session.client("sts").get_caller_identity()["Account"]

# アカウントリージョン名前空間のバケット名(末尾 -an は必須サフィックス)
BUCKET = f"{BUCKET_PREFIX}-{account_id}-{REGION}-an"
print("bucket:", BUCKET)

keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=LANDING_PREFIX):
    keys += [o["Key"] for o in page.get("Contents", []) if o["Key"].endswith(".csv")]
keys.sort()
print(f"landing の CSV: {len(keys)} 本")
for k in keys:
    print("  ", k)
assert keys, "landing に CSV が無い"

## 1〜3. 文字コード・改行コードをそろえて構造を確認する

- 文字コード: BOM 付き UTF-8 → UTF-8にデコードをする
- 改行コード: CRLF と CR を LF に置き換える。
- 構造: `csv` モジュールで1行ずつ読み、次を問題として数える。
  - 列数がヘッダと違う行
  - 空行(途中・末尾とも)
  - 1列目が年月日コード(10桁の数字)でない行 = 注釈行・出典行の疑い
  - 囲み文字 `"` を含む行。bronze を読む Athena の LazySimpleSerDe は `"` を囲み文字として
    扱わないので、`"秋田市,北部"` のような値は bronze で列がずれる。ここで止める

In [ ]:
import csv
import io
import re

def decode(raw: bytes) -> tuple[str, str]:
    if raw.startswith(b"\xef\xbb\xbf"):
        return raw[3:].decode("utf-8"), "UTF-8 (BOMあり)"
    # UTF-8 以外(CP932 など)は変換しない。UnicodeDecodeError のまま失敗させる
    return raw.decode("utf-8"), "UTF-8"

def newline_kind(raw: bytes) -> str:
    crlf = raw.count(b"\r\n")
    lf = raw.count(b"\n") - crlf
    cr = raw.count(b"\r") - crlf
    kinds = [n for n, c in (("CRLF", crlf), ("LF", lf), ("CR", cr)) if c]
    return "+".join(kinds) or "なし"

def normalize_and_check(key: str) -> dict:
    raw = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    result = {"key": key, "bytes_in": len(raw), "newline_in": newline_kind(raw), "problems": []}
    try:
        text, result["encoding_in"] = decode(raw)
    except UnicodeDecodeError as e:
        result["encoding_in"] = "不明"
        result["problems"].append(f"UTF-8 ではない(変換しない): {e}")
        return result

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")
    if lines and lines[-1] == "":
        lines.pop()  # ファイル末尾の改行1つぶん

    # 囲み文字 " は bronze(LazySimpleSerDe)で正しく読めないので、見つけたら止める。
    # csv.reader に渡す前に生の行で見る(囲みの中に改行があると行の分け方自体が変わるため)。
    for lineno, line in enumerate(lines, start=1):
        if '"' in line:
            result["problems"].append(f"{lineno}行目: 囲み文字 \" を含む {line[:40]!r}")

    rows = list(csv.reader(lines))
    header, body = rows[0], rows[1:]
    ncols = len(header)
    key_re = re.compile(KEY_PATTERN)
    for lineno, row in enumerate(body, start=2):
        if not row or all(c.strip() == "" for c in row):
            result["problems"].append(f"{lineno}行目: 空行")
        elif len(row) != ncols:
            result["problems"].append(f"{lineno}行目: 列数 {len(row)}(ヘッダは {ncols})")
        elif not key_re.match(row[0].strip()):
            result["problems"].append(f"{lineno}行目: 注釈行の疑い 1列目={row[0][:20]!r}")

    out = io.StringIO()
    csv.writer(out, lineterminator="\n").writerows(rows)
    result.update(
        columns=ncols,
        rows=len(body),
        header_tail=header[-1],
        normalized=out.getvalue().encode("utf-8"),
    )
    return result

results = [normalize_and_check(k) for k in keys]

print(f"{'ファイル':<32}{'文字コード':<16}{'改行':<6}{'列':>5}{'行':>6}  問題")
for r in results:
    name = r["key"].rsplit("/", 1)[-1]
    print(f"{name:<32}{r['encoding_in']:<16}{r['newline_in']:<6}{r.get('columns', '-'):>5}{r.get('rows', '-'):>6}  {len(r['problems'])}")
    for p in r["problems"][:10]:
        print("     -", p)
    if len(r["problems"]) > 10:
        print(f"     … ほか {len(r['problems']) - 10} 件")

failed = [r for r in results if r["problems"]]
print()
print("列数の分布:", sorted({r.get("columns") for r in results if "columns" in r}))
print("確認結果:", "OK" if not failed else f"NG {len(failed)} 本")

## 4. bronze に書き出す

全ファイルが確認を通ったときだけ書く(1本でも問題があれば何も書かない)。
出力は UTF-8(BOMなし)・LF・ファイル名は landing と同じ。

途中で失敗しても bronze に新旧が混ざらないよう、3段階で書く。

1. **一時置き場に全部書く** — `_staging/<実行時刻>/` に9本を置く。bronze は触らない
2. **一時置き場を読み直して検証する** — 9本そろっているか、中身が書いたものと一致するか
3. **bronze を truncate してからコピーする** — bronze の場所にあるファイルを全部消し、
   S3 内のコピー(`copy_object`)で書き込む。最後に一時置き場を消す

truncate で、landing から消えたファイルの残りも bronze から消える(Athena は bronze の場所に
あるファイルを全部読むので、残っていると古いデータが混ざるため)。消したファイルは report に残す。
書き出した後に、bronze のファイル一覧が今回のファイルとちょうど一致することを確かめる。

1〜2 で失敗した場合、bronze は前回のまま変わらない。一時置き場は調査用に残す。
3 は truncate してから書くので、終わるまでの間 bronze は空か一部だけの状態になる。
3 の途中で失敗した場合は、bronze が一部だけの状態で残る(S3 には複数オブジェクトを
まとめて入れ替える操作が無いため)。一時置き場は残っていて、入力が同じなら同じ結果になるので、
**ノートブックをもう一度実行すれば揃う**。

In [ ]:
import json
from datetime import datetime, timezone

if failed:
    raise SystemExit(f"確認に失敗したファイルがあるので bronze には書かない: {[r['key'] for r in failed]}")

ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
staging = f"{STAGING_PREFIX}{ts}/"
names = [r["key"].rsplit("/", 1)[-1] for r in results]

def bronze_files() -> list[str]:
    return sorted(
        o["Key"][len(BRONZE_PREFIX):]
        for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=BRONZE_PREFIX)
        for o in page.get("Contents", [])
    )

# landing に無いファイル(landing から消えたものの残り)。truncate で消えるので記録だけする
extra = sorted(set(bronze_files()) - set(names))
if extra:
    print(f"bronze にだけあるファイル(truncate で消える): {extra}")
report = {
    "run_at": ts,
    "bucket": BUCKET,
    "source": LANDING_PREFIX,
    "target": BRONZE_PREFIX,
    "staging": staging,
    "dry_run": DRY_RUN,
    "files": [{k: v for k, v in r.items() if k != "normalized"} | {"bytes_out": len(r["normalized"])} for r in results],
}

def put_report(status: str, **extra) -> None:
    if DRY_RUN:
        return
    body = report | {"status": status, **extra}
    key = f"{REPORT_PREFIX}{ts}.json"
    s3.put_object(Bucket=BUCKET, Key=key, ContentType="application/json",
                  Body=json.dumps(body, ensure_ascii=False, indent=2).encode("utf-8"))
    print(f"report: s3://{BUCKET}/{key}  status={status}")

# ── (1) 一時置き場に全部書く ────────────────────────────────────────────────
for r, name in zip(results, names):
    key = staging + name
    if DRY_RUN:
        print(f"[dry-run] {key}  {r['bytes_in']:,} -> {len(r['normalized']):,} bytes")
        continue
    s3.put_object(
        Bucket=BUCKET,
        Key=key,
        Body=r["normalized"],
        ContentType="text/csv; charset=utf-8",
        Metadata={"source-key": r["key"], "encoding-in": r["encoding_in"].encode("ascii", "ignore").decode() or "unknown"},
    )
    print(f"staged s3://{BUCKET}/{key}  {len(r['normalized']):,} bytes")

In [ ]:
# ── (2) 一時置き場を読み直して検証する ──────────────────────────────────────
if not DRY_RUN:
    try:
        staged = sorted(
            o["Key"].rsplit("/", 1)[-1]
            for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=staging)
            for o in page.get("Contents", [])
        )
        assert staged == sorted(names), f"一時置き場のファイルがそろっていない: {staged}"
        for r, name in zip(results, names):
            b = s3.get_object(Bucket=BUCKET, Key=staging + name)["Body"].read()
            assert b == r["normalized"], f"中身が一致しない: {name}"
    except Exception as e:
        put_report("failed_staging", error=str(e))
        raise SystemExit(f"一時置き場の検証に失敗。bronze は変更していない: {e}")
    print("一時置き場の検証 OK")

In [ ]:
# ── (3) bronze を truncate してからコピーし、一時置き場を消す ──────────────────
if DRY_RUN:
    print(f"[dry-run] truncate s3://{BUCKET}/{BRONZE_PREFIX}  {len(bronze_files())} 本を消す")
if not DRY_RUN:
    # truncate: bronze の場所にあるファイルを全部消す
    removed = bronze_files()
    for i in range(0, len(removed), 1000):
        s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": BRONZE_PREFIX + n} for n in removed[i:i + 1000]]})
    assert not bronze_files(), "truncate 後に bronze にファイルが残っている"
    report["truncated"] = removed
    print(f"truncated s3://{BUCKET}/{BRONZE_PREFIX}  {len(removed)} 本")

    done = []
    try:
        for name in names:
            s3.copy_object(
                Bucket=BUCKET,
                Key=BRONZE_PREFIX + name,
                CopySource={"Bucket": BUCKET, "Key": staging + name},
                MetadataDirective="COPY",
            )
            done.append(name)
            print(f"wrote s3://{BUCKET}/{BRONZE_PREFIX}{name}")
    except Exception as e:
        put_report("failed_copy", error=str(e), copied=done)
        raise SystemExit(f"bronze へのコピーが途中で失敗({len(done)}/{len(names)} 本)。もう一度実行すれば揃う: {e}")

    # bronze を読み直して、一時置き場と同じものになっているか確かめる
    for r, name in zip(results, names):
        b = s3.get_object(Bucket=BUCKET, Key=BRONZE_PREFIX + name)["Body"].read()
        assert b == r["normalized"], f"bronze の中身が一致しない: {name}"
    print("bronze の読み直し OK")

    # bronze のファイル一覧が今回のファイルとちょうど一致するか(過不足なし)
    now = bronze_files()
    if now != sorted(names):
        put_report("failed_file_list", bronze_files=now)
        raise SystemExit(f"bronze のファイル一覧が一致しない: 余分 {sorted(set(now) - set(names))} / 不足 {sorted(set(names) - set(now))}")
    print(f"bronze のファイル一覧 OK({len(now)} 本)")

    s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": staging + n} for n in names]})
    put_report("succeeded")